# OPT-09 · IA-2: the full project, run from start to end
**Introduction to Optimization Techniques (IA-2)** · Dept. of IT, K. J. Somaiya College of Engineering
**Group OPT-09:** Razzan Tambe (15) · Saurabh Tripathi (04) · Vivek Sharma (12)

This notebook runs the whole project:

| Step | What happens |
|---|---|
| 1 | get the code and data (Colab clones the GitHub repo) |
| 2 | load all **11 af-series instances**, build the qq-model, compute the IA-2 lower bound |
| 3 | **walk-through**: one IA-1 run and one IA-2 run on af-2, step by step |
| 4 | **the full experiment**: IA-1 (baseline) and IA-2 (proposed) on all 11 instances |
| 5 | analysis: comparison tables (paper vs IA-1 vs IA-2), significance tests, Excel workbook, all graphs |
| 6 | check against the stored results of the paper |
| 7 | download everything that was produced |

* **IA-1 (baseline):** our IA-1 code, unchanged (`qq_model.py`, `slp.py`): qq-formulation, filter SLP and random multistart, after Grothey & McKinnon (2023).
* **IA-2 (proposed):** RG-ILS-SLP (`rg_ils.py`), with diet-LP relaxation, recipe-clustering start, LP repair/polish and a regret-guided iterated local search.

**Choose the size of the run in Step 4** (times are for Colab's 2 CPUs):

| MODE | what runs | time |
|---|---|---|
| `"quick"` | 11 instances × IA-1 + IA-2 × 5 runs: checks that the whole pipeline works | ≈ 15 min |
| `"seed"` | 11 instances × IA-1 + IA-2 + 2 ablations × 20 runs (seed 12345): **shows the real IA-1 vs IA-2 difference** | ≈ 1.5 h |
| `"full"` | the paper: 11 instances × 3 seeds × 20 runs + ablations | ≈ 4 h |

*Why quick mode shows only a small difference:* IA-2 spends its first 3 runs on fresh recipe starts, and its gains come from the guided runs that follow. With 5 runs only 2 guided runs happen, so use `"seed"` (20 runs) to see the effect reported in the paper. Step 6 always shows the stored full results as well.

Then choose **Runtime → Run all**. For `seed` and `full`, keep the Colab tab open, and mount Google Drive in Step 4 so the results are kept.

## Step 1 · Get the code and data

In [ ]:
import os, sys, subprocess, time
REPO_URL = "https://github.com/YOUR-USERNAME/OPT-09-IA2-Pooling-SLP.git"   # <- your GitHub repository
if os.path.exists("04_Source_Code"):              # notebook opened inside the project folder
    ROOT = os.path.abspath(".")
elif os.path.exists("../04_Source_Code"):
    ROOT = os.path.abspath("..")
else:                                             # Google Colab: clone the repository
    subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
    ROOT = os.path.abspath("repo")
CODE, DATA, STORED = (os.path.join(ROOT, d) for d in ("04_Source_Code", "05_Dataset", "06_Results"))
os.chdir(CODE); sys.path.insert(0, CODE)
import scipy
if tuple(int(v) for v in scipy.__version__.split(".")[:2]) < (1, 15):   # HiGHS LP solver is bundled from 1.15
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy>=1.15"], check=True)
    raise SystemExit("SciPy was upgraded: use Runtime -> Restart session, then Run all again")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("project folder:", ROOT)
print("code:", sorted(f for f in os.listdir(CODE) if f.endswith(".py")))
print("data:", sorted(os.listdir(DATA)))
print("numpy", np.__version__, "| scipy", scipy.__version__, "| pandas", pd.__version__, "| CPUs", os.cpu_count())

## Step 2 · The 11 instances, the qq-model and the IA-2 lower bound
`load_instance` and `QQModel` come from IA-1 and are unchanged. `diet_relaxation` is new in IA-2: one small LP per product, which gives a valid lower bound on the best possible cost.

In [ ]:
from qq_model import load_instance, QQModel
from rg_ils import diet_relaxation
from run_experiments import BEST_KNOWN, PAPER
BARON_LB = {"af-1": 295.93153, "af-2": 235751.11, "af-3": 120498.73, "af-4": 183620.39, "af-5": 123243.60,
            "af-5-fs": 118763.29, "af-6-f": 2937.68, "af-6-fs": 2881.41, "af-7": 145928.01, "af-7b": 145928.01,
            "af-7b-fs": 130147.66}
INSTANCES = ["af-1", "af-2", "af-3", "af-4", "af-5", "af-5-fs", "af-6-f", "af-6-fs", "af-7", "af-7b", "af-7b-fs"]
rows = []
for nm in INSTANCES:
    t0 = time.perf_counter()
    inst = load_instance(os.path.join(DATA, nm + ".txt")); mod = QQModel(inst)
    lb = diet_relaxation(mod)["LB"]
    rows.append(dict(instance=nm, nutrients=inst.N, raw_materials=inst.I, bins=inst.M, products=inst.P,
                     variables=mod.n, constraints=mod.m_total, best_known=BEST_KNOWN[nm],
                     IA2_diet_LB=round(lb, 2), Baron_LB=BARON_LB[nm], paper_pct_good=PAPER[nm]["good"],
                     LB_time_s=round(time.perf_counter() - t0, 2)))
instances_table = pd.DataFrame(rows)
instances_table

## Step 3 · Walk-through on af-2: one IA-1 run and one IA-2 run, step by step
On af-2 the published SLP is Good (within 0.2% of the best known cost) in only 19.4% of its runs.

In [ ]:
from slp import FilterSLP, SLPOptions
import rg_ils
nm = "af-2"; bk = BEST_KNOWN[nm]
mod = QQModel(load_instance(os.path.join(DATA, nm + ".txt")))
gap = lambda f: 100 * (f - bk) / bk
solver = FilterSLP(mod, SLPOptions())          # the IA-1 filter-SLP, unchanged, used by BOTH methods

print("=== IA-1 (baseline): random start -> filter SLP")
rng = np.random.default_rng(12345)
for k in range(3):
    x0 = mod.random_start(rng)                 # IA-1 random starting point
    r = solver.solve(x0)
    print(f"run {k}: start cost {mod.objective(x0):11.2f} (violation {mod.violation(x0)[1]:.1e}) -> "
          f"SLP {r.iters:3d} iterations -> cost {r.f:.2f}, feasible={r.feasible}, gap {gap(r.f):+.3f}%")

In [ ]:
print("=== IA-2 (proposed), step by step")
rng = np.random.default_rng(12345)
starter = rg_ils.RecipeStarter(mod)                                   # step 1: diet LP relaxation
print(f"1. diet LP lower bound {starter.diet['LB']:.2f}  (best known {bk:.2f}, Baron LB {BARON_LB[nm]:.2f})")
x0, v = starter.start(rng)                                            # steps 2-3: recipe start + LP repair
print(f"2-3. recipe-clustering start + LP repair: cost {mod.objective(x0):.2f}, violation {rg_ils.hinf(mod, x0):.1e}")
r = solver.solve(x0)                                                  # step 4: the same IA-1 SLP
print(f"4. filter SLP: {r.iters} iterations -> cost {r.f:.2f} (gap {gap(r.f):+.3f}%)")
xb = rg_ils.polish(mod, r.x)                                          # step 5: LP polish
print(f"5. LP polish -> cost {mod.objective(xb):.2f} (gap {gap(mod.objective(xb)):+.3f}%), violation {rg_ils.hinf(mod, xb):.1e}")
reg = rg_ils.regret(mod, xb, starter.diet)
print(f"6. regret: products paying most for bin sharing: {[int(k) for k in np.argsort(-reg)[:3]]} "
      f"(regret {np.sort(reg)[::-1][:3].round(1)})")

In [ ]:
print("=== IA-2 search: 8 runs (3 recipe starts, then regret-guided ILS runs with 4 SLP probes each)")
rng = np.random.default_rng(12345)
starter = rg_ils.RecipeStarter(mod)
log = lambda rc: print(f"run {rc.run}: {rc.kind:4s} cost {rc.f:.2f}  gap {gap(rc.f):+.3f}%  feasible={bool(rc.feasible)}  "
                       f"{'GOOD' if rc.feasible and gap(rc.f) <= 0.2 else '    '}  best so far {gap(rc.incumbent):+.3f}%  ({rc.time:.1f}s)")
recs, best_x = rg_ils.rg_ils(mod, solver, starter, rng, runs=8, log=log)

## Step 4 · The full experiment: IA-1 and IA-2 on all 11 instances
`run_all.py` runs `run_experiments.py --method baseline|proposed|abl_*` for every instance and seed, 2 processes in parallel. It then calls `analyze_results.py`, which builds the tables, tests and graphs. A short summary prints as each task finishes, and the full log of every task is in `results_rerun/logs/`.

In [ ]:
MODE = "quick"          # "quick" (~15 min) | "seed" (~1.5 h, = seed 12345 of the paper) | "full" (~4 h, = the paper)
SAVE_TO_DRIVE = False   # True on Colab: results go to Google Drive and survive a disconnect

OUT = os.path.join(ROOT, "results_rerun")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/OPT-09_IA2_results_rerun"
t0 = time.time()
p = subprocess.Popen([sys.executable, "-u", "run_all.py", "--mode", MODE, "--out", OUT, "--jobs", "2"],
                     cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
print(f"\nfinished in {(time.time() - t0) / 60:.1f} min, return code {p.returncode}; results in {OUT}")

## Step 5 · Results of this run: paper vs IA-1 vs IA-2

In [ ]:
cmp = pd.read_csv(os.path.join(OUT, "comparison.csv"))
table = cmp[["instance", "paper_good", "base_feas", "prop_feas", "base_good", "prop_good",
             "paper_ETiGood", "base_ETiGood", "prop_ETiGood", "base_best_gap_pct", "prop_best_gap_pct"]].rename(columns={
    "paper_good": "paper %Good", "base_feas": "IA-1 %Feas", "prop_feas": "IA-2 %Feas", "base_good": "IA-1 %Good",
    "prop_good": "IA-2 %Good", "paper_ETiGood": "paper ETiGood", "base_ETiGood": "IA-1 ETiGood",
    "prop_ETiGood": "IA-2 ETiGood", "base_best_gap_pct": "IA-1 best gap %", "prop_best_gap_pct": "IA-2 best gap %"}).round(2)
print(f"mean over {len(cmp)} instances:  %Feas IA-1 {cmp.base_feas.mean():.1f} -> IA-2 {cmp.prop_feas.mean():.1f}   |   "
      f"%Good IA-1 {cmp.base_good.mean():.1f} -> IA-2 {cmp.prop_good.mean():.1f}")
table

In [ ]:
tests = pd.read_csv(os.path.join(OUT, "stats.csv"))
tests = tests.rename(columns={"good_base": "Good IA-1", "good_prop": "Good IA-2", "feas_base": "Feas IA-1",
                              "feas_prop": "Feas IA-2", "median_gap_base": "median gap IA-1 %",
                              "median_gap_prop": "median gap IA-2 %"}).round(4)
tests

In [ ]:
from IPython.display import Image, display
for f in ["fig_pct_good.png", "fig_pct_feas.png", "fig_etigood.png", "fig_convergence.png", "fig_gaps.png", "fig_ablation.png"]:
    path = os.path.join(OUT, "Graphs", f)
    if os.path.exists(path):
        print(f); display(Image(path, width=900))
print("Excel workbook with all tables:", os.path.join(OUT, "Result_Tables.xlsx"))

## Step 6 · Check against the stored results of the paper
`06_Results` holds the paper's experiment: 3 seeds × 20 runs per method. Every run uses a fixed seed, so a rerun gives the **same costs**; only the times depend on the machine. For seed 12345 the table below compares your rerun with the stored runs, run by run.

The exception is af-7b-fs, the largest instance. The IA-1 SLP stops any single LP after 3 s, and on a slower or busier machine a few LPs hit that limit, so some runs take a slightly different path (in our test, cost differences of at most 0.015%, all still Good).

In [ ]:
def load_runs(folder, method, seed=12345):
    d = os.path.join(folder, method, f"seed{seed}")
    fs = [f for f in os.listdir(d) if f.startswith("runs_") and f.endswith(".csv")] if os.path.isdir(d) else []
    return pd.concat([pd.read_csv(os.path.join(d, f)) for f in fs]) if fs else pd.DataFrame()
check = []
for m in ["baseline", "proposed"]:
    new, old = load_runs(OUT, m), load_runs(STORED, m)
    if new.empty or old.empty:
        continue
    j = new.merge(old, on=["instance", "run"], suffixes=("_new", "_stored"))
    for nm, g in j.groupby("instance"):
        check.append(dict(method="IA-1" if m == "baseline" else "IA-2", instance=nm, runs_compared=len(g),
                          identical_costs=int((abs(g.f_new - g.f_stored) <= 1e-6 * abs(g.f_stored)).sum())))
pd.DataFrame(check)

In [ ]:
stored = pd.read_csv(os.path.join(STORED, "comparison.csv"))
print("stored paper results (3 seeds x 20 runs per method):")
print(f"mean %Feas IA-1 {stored.base_feas.mean():.1f} -> IA-2 {stored.prop_feas.mean():.1f}  |  "
      f"mean %Good IA-1 {stored.base_good.mean():.1f} -> IA-2 {stored.prop_good.mean():.1f}")
stored[["instance", "paper_good", "base_good", "prop_good", "base_feas", "prop_feas"]].rename(columns={
    "paper_good": "paper %Good", "base_good": "IA-1 %Good", "prop_good": "IA-2 %Good",
    "base_feas": "IA-1 %Feas", "prop_feas": "IA-2 %Feas"}).round(1)

## Step 7 · Download what this run produced

In [ ]:
import shutil
archive = shutil.make_archive(os.path.join(ROOT, "results_rerun_" + MODE), "zip", OUT)
print("zipped:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("(not on Colab: the zip is in the project folder)")